# T04 — DAVIS Utilities + Metrics Check

Checks `anyup/davis.py` and `anyup/metrics.py` (task T04, see `docs/plan/P1_experiments/T04_davis_metrics.md`) on real DAVIS-2017 data.

1. Vendored DAVIS J/F = official `davis2017-evaluation` package (exact), on GT vs perturbed masks
2. `propagate_labels` = Exp 3 `propagate_clip` (exact, CPU)
3. Full official evaluation: bilinear Swin-T → propagation on all 30 val clips → our `summarize_jf` = package `DAVISEvaluation` (exact)
4. Old Exp 3 metric vs official on the same masks (INFO)
5. Temporal cosine (float64 vs Exp 2 float32, chunk-boundary split) and GAP fidelity on `bear`, VideoMAE (INFO)

T4 is enough (AnyUp3D at 112² with `q_chunk_size=4096`). **Runtime → Restart session** before running if `anyup` was imported before.

In [ ]:
# ── Cell 1 · Repo + Drive + official DAVIS evaluation package ───────────────
import os, sys, subprocess
from google.colab import drive
drive.mount("/content/drive")

REPO_URL    = "https://github.com/mu-az88/anyup.git"
REPO_BRANCH = "experiments-task02"   # T04 continues on the T02 branch
REPO_DIR    = "/content/anyup"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-b", REPO_BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    # existing clone may be on another branch — switch to REPO_BRANCH and update it
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "-B", REPO_BRANCH, f"origin/{REPO_BRANCH}"], check=True)
if "anyup" in sys.modules:
    raise RuntimeError("anyup was already imported in this runtime — Runtime → Restart session, then run again")
subprocess.run(["pip", "install", "-e", REPO_DIR, "-q"], check=True)
subprocess.run(["pip", "install", "transformers", "timm", "scikit-image", "scipy", "-q"], check=True)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Official toolkit at the commit the vendored code was copied from (T04 log)
DAVIS_EVAL_DIR    = "/content/davis2017-evaluation"
DAVIS_EVAL_COMMIT = "ac7c43fca936f9722837b7fbd337d284ba37004b"
if not os.path.exists(DAVIS_EVAL_DIR):
    subprocess.run(["git", "clone", "https://github.com/davisvideochallenge/davis2017-evaluation.git", DAVIS_EVAL_DIR], check=True)
subprocess.run(["git", "-C", DAVIS_EVAL_DIR, "checkout", DAVIS_EVAL_COMMIT], check=True)
subprocess.run(["pip", "install", DAVIS_EVAL_DIR, "-q"], check=True)
print(subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
# ── Cell 2 · Imports + config ───────────────────────────────────────────────
import json, shutil, time
from pathlib import Path
import numpy as np
import cv2
import torch
import torch.nn.functional as F
from PIL import Image

from anyup import davis, metrics
from anyup.backbones import BackboneLoader
from anyup.video import iter_backbone_chunks, chunk_boundaries
from anyup.upsamplers import UpsamplerLoader

from davis2017 import metrics as dm, utils as du          # official package
from davis2017.evaluation import DAVISEvaluation

DEVICE        = "cuda"
CKPT_ANYUP3D  = "/content/drive/MyDrive/results/anyup3d_step6500.pth"   # only for the INFO checks in Cell 7 (as T03)
PROP_SIZE     = 56      # ↓ Exp 3 propagation grid = bilinear output size; affinity ∝ PROP_SIZE⁴. Dependent: Cells 4, 5
N_VENDOR_CLIPS = 3      # ↓ val clips used for the vendored-vs-package check (Cell 3); all frames each
TEMP_CLIP     = "bear"  # 82 frames (DAVIS-2017 train), as T02/T03
GUIDE_SIZE    = 112     # ↓ guide = AnyUp output size for Cell 7 (T4-safe, as T03 reduced run). Dependent: Q_CHUNK, CPU RAM (41·768·GUIDE_SIZE²·4 B per method)
Q_CHUNK       = 4096    # ↓ AnyUp3D query chunk for Cell 7; smaller = less GPU memory
EXACT_TOL     = 1e-12   # vendored/our aggregation vs package: everything is the same float64 code → expect 0
RES_DIR       = Path("/content/t04_results/bilinear")       # predicted PNGs, official results layout <clip>/<stem>.png
OUT_DIR       = "/content/drive/MyDrive/results/T04_davis_metrics_check"   # save_results output (Cell 8)

RESULTS = []   # (check name, passed: bool | None for informational, detail str)
def record(name, passed, detail):
    RESULTS.append((name, passed, detail))
    tag = "INFO" if passed is None else ("PASS" if passed else "FAIL")
    print(f"[{tag}] {name}: {detail}")

print("Device:", DEVICE, torch.cuda.get_device_name(0))

In [ ]:
# ── Cell 3 · DAVIS (ETH) + vendored J/F == official package ────────────────
ROOT  = davis.download_davis("/content")
CLIPS = davis.list_clips(ROOT, "val")
record("val clips", len(CLIPS) == 30, f"n={len(CLIPS)} first={CLIPS[:3]}")

def variants(gt0, n_obj, rng):
    """Predictions to score against gt0 (N, H, W): identical, shifted, eroded, last object dropped, 2% noise."""
    noise = gt0.copy()
    m = rng.random(gt0.shape) < 0.02
    noise[m] = rng.integers(0, n_obj + 1, m.sum())
    dropped = gt0.copy(); dropped[dropped == n_obj] = 0
    eroded = np.stack([cv2.erode(f, np.ones((7, 7), np.uint8)) for f in gt0])
    return {"identical": gt0, "shift5": np.roll(gt0, 5, axis=2), "erode7": eroded,
            "drop_last": dropped, "noise2pct": noise}

rng = np.random.default_rng(0)
max_diff, n_cmp = 0.0, 0
for clip in CLIPS[:N_VENDOR_CLIPS]:
    gt = davis.load_annotations(ROOT, clip)
    gt0 = gt.copy(); gt0[gt0 == 255] = 0
    n_obj = davis.num_objects(gt[0])
    g_oh = davis.one_hot_objects(gt0, n_obj)                      # (n_obj, N, H, W)
    for vname, pred in variants(gt0, n_obj, rng).items():
        p_oh = davis.one_hot_objects(pred, n_obj)
        for k in range(n_obj):
            ours = (metrics.db_eval_iou(g_oh[k], p_oh[k]), metrics.db_eval_boundary(g_oh[k], p_oh[k]))
            pkg  = (dm.db_eval_iou(g_oh[k], p_oh[k]),      dm.db_eval_boundary(g_oh[k], p_oh[k]))
            stats_o = metrics.db_statistics(ours[0]) + metrics.db_statistics(ours[1])
            stats_p = du.db_statistics(pkg[0]) + du.db_statistics(pkg[1])
            d = max(np.abs(ours[0] - pkg[0]).max(), np.abs(ours[1] - pkg[1]).max(),
                    np.abs(np.array(stats_o) - np.array(stats_p)).max())
            max_diff = max(max_diff, float(d)); n_cmp += 1
    print(f"  {clip}: {gt.shape[0]} frames, {n_obj} objects")
record("vendored J/F/statistics == package", max_diff == 0,
       f"clips={CLIPS[:N_VENDOR_CLIPS]} variants=5 object-comparisons={n_cmp} max_abs_diff={max_diff:.2e}")

In [ ]:
# ── Cell 4 · Swin-T bilinear features + propagate_labels == Exp 3 (exact) ──
bb   = BackboneLoader.load("swin_t", device=DEVICE)      # per_frame (Exp 3 frame-level features), T01 default norm
upbl = UpsamplerLoader.load("bilinear", device=DEVICE)

def bilinear_feats(clip):
    """All frames → (N, 768, PROP_SIZE, PROP_SIZE) float32 on CPU (Exp 3 setting, all frames as T02 Q2)."""
    out = [upbl(ch, output_size=PROP_SIZE).features.cpu() for ch in iter_backbone_chunks(bb, davis.open_clip(ROOT, clip))]
    return torch.cat(out)

# Exp 3 Cell 2 constants and Cell 5/7 functions, verbatim (comments trimmed)
TAU, TOP_K, N_LAST_FRAMES = 0.07, 7, 6
def load_ann(ann_path):
    return np.array(Image.open(ann_path))

def build_label_maps_frame0(ann0_HW, n_objects, H, W):
    ann_resized = cv2.resize(
        ann0_HW, (W, H), interpolation=cv2.INTER_NEAREST
    ).astype(np.int64)
    n_ch    = n_objects + 1
    N       = H * W
    label   = np.zeros((n_ch, N), dtype=np.float32)
    ann_flat = ann_resized.flatten()
    for obj_id in range(n_ch):
        label[obj_id, ann_flat == obj_id] = 1.0
    return torch.from_numpy(label)

@torch.no_grad()
def propagate_clip(feats_TCHW, ann_paths, sampled_idxs):
    T, C, H, W = feats_TCHW.shape
    N = H * W
    feats_flat = feats_TCHW.view(T, C, N).permute(0, 2, 1)
    feats_flat = F.normalize(feats_flat, dim=-1)
    ann0 = load_ann(ann_paths[sampled_idxs[0]])
    n_objects = int(ann0.max())
    n_ch = n_objects + 1
    label_maps = torch.zeros(T, n_ch, N)
    label_maps[0] = build_label_maps_frame0(ann0, n_objects, H, W)
    for t in range(1, T):
        past      = list(range(max(1, t - N_LAST_FRAMES), t))
        context_t = [0] + past
        ctx_feats  = feats_flat[context_t].reshape(-1, C)
        ctx_labels = label_maps[context_t].permute(1, 0, 2).reshape(n_ch, -1)
        q = feats_flat[t]
        A = torch.mm(q, ctx_feats.T) / TAU
        topk_vals, topk_idx = A.topk(TOP_K, dim=-1)
        A_sparse = torch.full_like(A, float("-inf"))
        A_sparse.scatter_(1, topk_idx, topk_vals)
        A_soft = F.softmax(A_sparse, dim=-1)
        new_labels = torch.mm(A_soft, ctx_labels.T)
        label_maps[t] = new_labels.T
    pred = label_maps.argmax(dim=1)
    return pred.view(T, H, W)

clip = CLIPS[0]
feats = bilinear_feats(clip)                                     # (N, 768, 56, 56) CPU
ap = davis.ann_paths(ROOT, clip)
t0 = time.perf_counter(); old = propagate_clip(feats, ap, list(range(feats.shape[0]))); t_old = time.perf_counter() - t0
t0 = time.perf_counter(); new = davis.propagate_labels(feats, davis.load_annotation(ap[0])); t_new = time.perf_counter() - t0
record("propagate_labels == Exp 3 propagate_clip (CPU)", torch.equal(old, new),
       f"clip={clip} feats={tuple(feats.shape)} pred={tuple(new.shape)} differing_px={(old != new).sum().item()} "
       f"time old/new={t_old:.1f}/{t_new:.1f}s")
t0 = time.perf_counter(); gpu = davis.propagate_labels(feats.to(DEVICE), davis.load_annotation(ap[0])); t_gpu = time.perf_counter() - t0
record("propagate_labels GPU vs CPU", None,
       f"differing_px={(gpu != new).sum().item()} / {new.numel()} ({(gpu != new).float().mean().item():.2e}) time={t_gpu:.1f}s")
del feats, old, new, gpu

In [ ]:
# ── Cell 5 · Full official evaluation: ours == DAVISEvaluation (exact) ─────
shutil.rmtree(RES_DIR, ignore_errors=True)
per_clip, t0 = {}, time.perf_counter()
for clip in CLIPS:
    feats = bilinear_feats(clip).to(DEVICE)                      # (N, 768, PROP_SIZE, PROP_SIZE)
    ap, fp = davis.ann_paths(ROOT, clip), davis.frame_paths(ROOT, clip)
    pred = davis.propagate_labels(feats, davis.load_annotation(ap[0]))   # (N, PROP_SIZE, PROP_SIZE), GPU (T05 path)
    for t, f in enumerate(fp):
        davis.save_mask_png(pred[t], ap[0], RES_DIR / clip / f"{f.stem}.png")
    pred_native = np.stack([np.array(Image.open(RES_DIR / clip / f"{f.stem}.png")) for f in fp])   # what the package reads
    per_clip[clip] = metrics.eval_clip_jf(davis.load_annotations(ROOT, clip), pred_native)
    del feats, pred; torch.cuda.empty_cache()
OURS = metrics.summarize_jf(per_clip)
print(f"propagation + our eval: {time.perf_counter() - t0:.0f}s")

t0 = time.perf_counter()
PKG = DAVISEvaluation(davis_root=str(ROOT), task="semi-supervised", gt_set="val").evaluate(str(RES_DIR))
print(f"package eval: {time.perf_counter() - t0:.0f}s")

ours_lists = {m: {s: [v for r in per_clip.values() for v in r[f"{m}_{s}"]] for s in "MRD"} for m in "JF"}
diffs = {f"{m}_{s}": float(np.max(np.abs(np.array(ours_lists[m][s]) - np.array(PKG[m][s])))) for m in "JF" for s in "MRD"}
keys_ok = list(OURS["per_object"]) == list(PKG["J"]["M_per_object"]) == list(PKG["F"]["M_per_object"])
obj_diff = max(max(abs(OURS["per_object"][k]["J"] - PKG["J"]["M_per_object"][k]),
                   abs(OURS["per_object"][k]["F"] - PKG["F"]["M_per_object"][k])) for k in OURS["per_object"])
pkg_global = {"J&F-Mean": (np.mean(PKG["J"]["M"]) + np.mean(PKG["F"]["M"])) / 2, "J-Mean": np.mean(PKG["J"]["M"]),
              "J-Recall": np.mean(PKG["J"]["R"]), "J-Decay": np.mean(PKG["J"]["D"]), "F-Mean": np.mean(PKG["F"]["M"]),
              "F-Recall": np.mean(PKG["F"]["R"]), "F-Decay": np.mean(PKG["F"]["D"])}
g_diff = max(abs(OURS[k] - float(v)) for k, v in pkg_global.items())
ok = keys_ok and max(diffs.values()) < EXACT_TOL and obj_diff < EXACT_TOL and g_diff < EXACT_TOL
record("summarize_jf == DAVISEvaluation (30 val clips, bilinear Swin-T)", ok,
       f"objects={OURS['n_objects']} keys_equal={keys_ok} max_diff per-stat={max(diffs.values()):.2e} "
       f"per-object={obj_diff:.2e} global={g_diff:.2e}")
record("official J&F, bilinear Swin-T per-frame, PROP_SIZE=56, all frames", None,
       " ".join(f"{k}={OURS[k]:.4f}" for k in pkg_global))

In [ ]:
# ── Cell 6 · Old Exp 3 metric vs official on the same masks (INFO) ─────────
# Exp 3 Cell 8, verbatim (comments trimmed)
def compute_J(pred_HW, gt_HW):
    obj_ids = np.unique(gt_HW)
    obj_ids = obj_ids[obj_ids > 0]
    if len(obj_ids) == 0:
        return 1.0
    j_scores = []
    for obj_id in obj_ids:
        pred_fg = (pred_HW == obj_id)
        gt_fg   = (gt_HW   == obj_id)
        inter   = np.logical_and(pred_fg, gt_fg).sum()
        union   = np.logical_or(pred_fg,  gt_fg).sum()
        j_scores.append(inter / union if union > 0 else 1.0)
    return float(np.mean(j_scores))

def mask_to_boundary(mask_HW, dilation_px=2):
    mask_u8 = mask_HW.astype(np.uint8)
    kernel  = np.ones((dilation_px * 2 + 1, dilation_px * 2 + 1), np.uint8)
    eroded  = cv2.erode(mask_u8, kernel, iterations=1)
    return mask_u8 - eroded

def compute_F(pred_HW, gt_HW, bound_th=0.008, dilation_px=2):
    obj_ids = np.unique(gt_HW)
    obj_ids = obj_ids[obj_ids > 0]
    if len(obj_ids) == 0:
        return 1.0
    bound_pix = max(1, int(np.ceil(bound_th * np.linalg.norm(gt_HW.shape))))
    struct = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,
                                        (2 * bound_pix + 1, 2 * bound_pix + 1))
    f_scores = []
    for obj_id in obj_ids:
        pred_b = mask_to_boundary((pred_HW == obj_id).astype(np.uint8), dilation_px)
        gt_b   = mask_to_boundary((gt_HW   == obj_id).astype(np.uint8), dilation_px)
        n_pred, n_gt = pred_b.sum(), gt_b.sum()
        if n_pred == 0 and n_gt == 0:
            f_scores.append(1.0); continue
        if n_pred == 0 or n_gt == 0:
            f_scores.append(0.0); continue
        pred_dil = cv2.dilate(pred_b, struct)
        gt_dil   = cv2.dilate(gt_b,   struct)
        precision = np.logical_and(pred_b, gt_dil).sum() / (n_pred + 1e-8)
        recall    = np.logical_and(gt_b,   pred_dil).sum() / (n_gt   + 1e-8)
        f_scores.append(float(2 * precision * recall / (precision + recall + 1e-8)))
    return float(np.mean(f_scores))

def old_eval(skip_first):
    """Exp 3 evaluate_condition aggregation (frame-pooled), all frames; skip_first drops frame 0."""
    J_all, F_all = [], []
    for clip in CLIPS:
        gt = davis.load_annotations(ROOT, clip)
        fp = davis.frame_paths(ROOT, clip)
        for t in range(1 if skip_first else 0, len(fp)):
            pred = np.array(Image.open(RES_DIR / clip / f"{fp[t].stem}.png"))
            J_all.append(compute_J(pred, gt[t])); F_all.append(compute_F(pred, gt[t]))
    J, Fm = float(np.mean(J_all)), float(np.mean(F_all))
    return J, Fm, (J + Fm) / 2

for label, skip in [("old Exp 3 metric (frame 0 scored)", False), ("old Exp 3 metric, frame 0 dropped", True)]:
    J, Fm, JF = old_eval(skip)
    record(label, None, f"J={J:.4f} F={Fm:.4f} J&F={JF:.4f} | official J={OURS['J-Mean']:.4f} F={OURS['F-Mean']:.4f} "
                        f"J&F={OURS['J&F-Mean']:.4f} | Δ J&F (old − official)={JF - OURS['J&F-Mean']:+.4f}")
del bb; torch.cuda.empty_cache()

In [ ]:
# ── Cell 7 · Temporal cosine + GAP fidelity on bear, VideoMAE (INFO) ───────
def adjacent_cosine_sim(feats):          # Exp 2 Cell 6, verbatim (float32, whole tensor)
    feats_norm = F.normalize(feats, dim=1)
    sim        = (feats_norm[:-1] * feats_norm[1:]).sum(dim=1)
    return sim.mean().item()

bbv = BackboneLoader.load("videomae", device=DEVICE, guide_size=GUIDE_SIZE)
ups = {"bilinear": upbl,
       "anyup2d": UpsamplerLoader.load("anyup2d", device=DEVICE),
       "anyup3d": UpsamplerLoader.load("anyup3d", ckpt_path=CKPT_ANYUP3D, device=DEVICE, q_chunk_size=Q_CHUNK)}
src = davis.open_clip(ROOT, TEMP_CLIP)
for name, up in ups.items():
    lr, hr, metas = [], [], []
    for ch in iter_backbone_chunks(bbv, src):
        lr.append(ch.features.cpu()); res = up(ch); hr.append(res.features.cpu()); metas.append(res.meta)
        del res
    lr, hr = torch.cat(lr), torch.cat(hr)                           # (41, 768, 14, 14), (41, 768, GUIDE_SIZE, GUIDE_SIZE)
    bounds = chunk_boundaries(metas)
    tc = metrics.temporal_cosine(hr, bounds)
    old = adjacent_cosine_sim(hr)
    fid = metrics.gap_cosine_fidelity(lr, hr)
    record(f"temporal cosine videomae + {name}", None,
           f"all={tc['all']:.6f} within={tc['within']:.6f} cross={tc['cross']:.6f} (n_cross={tc['n_cross']}, bounds={bounds}) "
           f"| Exp 2 float32={old:.6f} diff={old - tc['all']:+.2e}")
    record(f"GAP fidelity videomae + {name}", None,
           f"mean={fid['mean']:.6f} min={min(fid['per_frame']):.6f} max={max(fid['per_frame']):.6f}")
    if name == "bilinear":
        tl = metrics.temporal_cosine(lr, bounds)
        record("temporal cosine videomae backbone features (14², reference)", None,
               f"all={tl['all']:.6f} within={tl['within']:.6f} cross={tl['cross']:.6f}")
    del lr, hr; torch.cuda.empty_cache()

## Summary — paste this output back

In [ ]:
# ── Cell 8 · Summary (saved to Drive with metrics.save_results) ─────────────
import transformers, timm, skimage, scipy
env = (f"torch={torch.__version__} torchvision={__import__('torchvision').__version__} transformers={transformers.__version__} "
       f"timm={timm.__version__} skimage={skimage.__version__} scipy={scipy.__version__} gpu={torch.cuda.get_device_name(0)}")
commit = subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--oneline"], capture_output=True, text=True).stdout.strip()
print(env); print("commit:", commit)
for name, passed, detail in RESULTS:
    tag = "INFO" if passed is None else ("PASS" if passed else "FAIL")
    print(f"[{tag}] {name}: {detail}")
n_fail = sum(p is False for _, p, _ in RESULTS)
print(f"\n{len(RESULTS)} checks, {n_fail} failed")

rows = [dict(object=k, **v) for k, v in OURS["per_object"].items()]
summary = dict(checks=[dict(name=n, passed=p, detail=d) for n, p, d in RESULTS], env_versions=env, commit=commit,
               official_bilinear={k: OURS[k] for k in ("J&F-Mean", "J-Mean", "J-Recall", "J-Decay", "F-Mean", "F-Recall", "F-Decay")})
config = dict(prop_size=PROP_SIZE, tau=TAU, top_k=TOP_K, n_last=N_LAST_FRAMES, backbone="swin_t per_frame", upsampler="bilinear",
              temp_clip=TEMP_CLIP, guide_size=GUIDE_SIZE, q_chunk=Q_CHUNK, ckpt=CKPT_ANYUP3D, davis_eval_commit=DAVIS_EVAL_COMMIT)
print("saved:", metrics.save_results(OUT_DIR, rows, summary, config, name="T04_check"))